In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("🚀 KHỞI ĐỘNG A/B TESTING FRAMEWORK (PRODUCTION MOCKUP)")
print("="*60)

# 1. Chuẩn bị dữ liệu và Huấn luyện 2 mô hình (Biased vs Fair)
df = pd.read_csv('../data/synthetic_intersectional_data.csv')
df['inter_group'] = df['gender'] + "_" + df['race']
features = ['age', 'education_years', 'income']

# Mô hình A (Biased)
model_A = LogisticRegression()
model_A.fit(df[features], df['loan_approved'])

# Mô hình B (Fair - Causal Intervention)
df_fair = df.copy()
global_mean = df_fair['income'].mean()
for g in df_fair['inter_group'].unique():
    g_mean = df_fair[df_fair['inter_group'] == g]['income'].mean()
    df_fair.loc[df_fair['inter_group'] == g, 'income'] = df_fair['income'] - g_mean + global_mean

model_B = LogisticRegression()
model_B.fit(df_fair[features], df_fair['loan_approved'])

# 2. Giả lập luồng Traffic Production (10,000 khách hàng mới)
print("Đang phân luồng 10,000 người dùng: 50% vào Group A (Cũ), 50% vào Group B (Mới)...")
np.random.seed(42)
traffic = df.sample(10000, random_state=42).copy()
traffic['ab_group'] = np.random.choice(['A', 'B'], size=len(traffic), p=[0.5, 0.5])

group_A = traffic[traffic['ab_group'] == 'A'].copy()
group_B = traffic[traffic['ab_group'] == 'B'].copy()

# 3. Chạy dự đoán
# Group A dùng dữ liệu gốc và model A
group_A['prediction'] = model_A.predict(group_A[features])

# Group B phải đi qua màng lọc can thiệp nhân quả trước khi vào model B
group_B_fair = group_B.copy()
for g in group_B_fair['inter_group'].unique():
    group_B_fair.loc[group_B_fair['inter_group'] == g, 'income'] = group_B_fair['income'] - g_mean + global_mean
group_B['prediction'] = model_B.predict(group_B_fair[features])

# 4. Đánh giá Causal Impact
def calc_metrics(data):
    acc = (data['prediction'] == data['loan_approved']).mean()
    approval_rates = data.groupby('inter_group')['prediction'].mean()
    dsc = approval_rates.min() / approval_rates.max()
    return acc, dsc

acc_A, dsc_A = calc_metrics(group_A)
acc_B, dsc_B = calc_metrics(group_B)

# Kiểm định thống kê (T-test) xem độ giảm Accuracy có đáng kể không
t_stat, p_val = stats.ttest_ind(
    (group_A['prediction'] == group_A['loan_approved']).astype(int),
    (group_B['prediction'] == group_B['loan_approved']).astype(int)
)

print("\n📊 KẾT QUẢ A/B TESTING:")
print(f"🔹 Nhóm A (Biased Model)  - Traffic: {len(group_A)} | Accuracy: {acc_A:.4f} | DSC: {dsc_A:.4f}")
print(f"🔸 Nhóm B (Fair Model)    - Traffic: {len(group_B)} | Accuracy: {acc_B:.4f} | DSC: {dsc_B:.4f}")

print("\n🎯 ƯỚC LƯỢNG TÁC ĐỘNG NHÂN QUẢ (CAUSAL IMPACT):")
print(f"👉 Tác động lên Công bằng (Fairness Impact): Cải thiện DSC lên {(dsc_B - dsc_A):.4f} điểm.")
print(f"👉 Sự đánh đổi Độ chính xác (Accuracy Trade-off): Giảm {(acc_A - acc_B)*100:.2f}%.")

if p_val > 0.05:
    print(f"✅ Kết luận Thống kê: Sự sụt giảm Accuracy là KHÔNG CÓ Ý NGHĨA THỐNG KÊ (p-value = {p_val:.4f} > 0.05).")
    print("🚀 ĐỀ XUẤT: An toàn để triển khai Mô hình B (Causal Fair Model) cho 100% người dùng!")
else:
    print(f"⚠️ Cảnh báo: Sự sụt giảm Accuracy là có ý nghĩa thống kê (p-value = {p_val:.4f} < 0.05). Cần tinh chỉnh thêm.")

🚀 KHỞI ĐỘNG A/B TESTING FRAMEWORK (PRODUCTION MOCKUP)
Đang phân luồng 10,000 người dùng: 50% vào Group A (Cũ), 50% vào Group B (Mới)...

📊 KẾT QUẢ A/B TESTING:
🔹 Nhóm A (Biased Model)  - Traffic: 5076 | Accuracy: 0.9007 | DSC: 0.9825
🔸 Nhóm B (Fair Model)    - Traffic: 4924 | Accuracy: 0.9023 | DSC: 0.9858

🎯 ƯỚC LƯỢNG TÁC ĐỘNG NHÂN QUẢ (CAUSAL IMPACT):
👉 Tác động lên Công bằng (Fairness Impact): Cải thiện DSC lên 0.0033 điểm.
👉 Sự đánh đổi Độ chính xác (Accuracy Trade-off): Giảm -0.16%.
✅ Kết luận Thống kê: Sự sụt giảm Accuracy là KHÔNG CÓ Ý NGHĨA THỐNG KÊ (p-value = 0.7876 > 0.05).
🚀 ĐỀ XUẤT: An toàn để triển khai Mô hình B (Causal Fair Model) cho 100% người dùng!
